In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# database
def get_data_tickers():
    url = "https://raw.githubusercontent.com/heyhanief/IDX-Data/main/Kompas100"
    try:
        df = pd.read_csv(url, header=None)
        tickers = df[0].astype(str).tolist()
        tickers = [t if t.endswith(".JK") else f"{t}.JK" for t in tickers]
        print(f"✓ Successfully imported {len(tickers)} tickers from Kompas100")
        return tickers
    except Exception as e:
        print(f"Failed to import tickers: {e}")
        return []

# Get tickers
print("\n" + "="*80)
print("STOCK MOMENTUM ANALYZER - KOMPAS100")
print("="*80)

tickers = get_data_tickers()
if not tickers:
    print("✗ No tickers found. Exiting.")
    exit(1)

# Period
start_date = "2024-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

# Download data with progress disabled and error handling
print(f"\n[1/4] Downloading historical data...")
try:
    data = yf.download(tickers, start=start_date, end=end_date, progress=False, group_by='column', auto_adjust=True)
except Exception as e:
    print(f"✗ Error downloading data: {e}")
    exit(1)

# Check if data is empty
if data.empty:
    print("✗ No data downloaded. Exiting.")
    exit(1)

# Extract Close and Volume data
try:
    prices = data["Close"]
    volumes = data["Volume"]
except KeyError:
    # Handle single ticker case
    if len(tickers) == 1:
        prices = data["Close"].to_frame()
        volumes = data["Volume"].to_frame()
        prices.columns = tickers
        volumes.columns = tickers
    else:
        raise

# Drop columns (tickers) with all NaN values
prices = prices.dropna(axis=1, how='all')
volumes = volumes.dropna(axis=1, how='all')

print(f"      ✓ Data retrieved for {len(prices.columns)} tickers")

# Check if we have enough historical data
lookback = 20  # 20 trading days
if len(prices) <= lookback:
    print(f"✗ Insufficient data. Need at least {lookback + 1} trading days, but only have {len(prices)}.")
    exit(1)

# Calculate 1-month momentum (only for tickers with sufficient data)
print(f"\n[2/4] Calculating momentum ({lookback}-day lookback)...")
momentum = pd.Series(dtype=float)
latest_price = pd.Series(dtype=float)
latest_volume = pd.Series(dtype=float)

for ticker in prices.columns:
    ticker_prices = prices[ticker].dropna()
    ticker_volumes = volumes[ticker].dropna()
    
    # Check if ticker has enough data points
    if len(ticker_prices) > lookback and len(ticker_volumes) > 0:
        momentum[ticker] = (ticker_prices.iloc[-1] / ticker_prices.iloc[-lookback]) - 1
        latest_price[ticker] = ticker_prices.iloc[-1]
        latest_volume[ticker] = ticker_volumes.iloc[-1]

print(f"      ✓ Momentum calculated for {len(momentum)} tickers")

# Filter Conditions
print(f"\n[3/4] Applying filters...")
print(f"      - Momentum: 0% to 15%")
print(f"      - Price: Rp 100 to Rp 1,000")
print(f"      - Volume: > 50,000")

cond1 = momentum > 0
cond2 = momentum <= 0.15
cond3 = latest_price.between(100, 1000)
cond4 = latest_volume > 50000

filtered = momentum[cond1 & cond2 & cond3 & cond4].dropna()

print(f"      ✓ {len(filtered)} stocks passed all filters")

# Check if we have any results
if len(filtered) == 0:
    print("\n✗ No stocks match the filter criteria.")
    print("="*80)
    exit(0)

# Sort results
print(f"\n[4/4] Sorting and analyzing results...")
sorted_result = filtered.sort_values(ascending=False)

# Calculate quartiles for momentum
quartiles = sorted_result.quantile([0.25, 0.5, 0.75])

# Create results dataframe
results_df = pd.DataFrame({
    'Ticker': sorted_result.index,
    'Momentum': sorted_result.values,
    'Price': [latest_price[ticker] for ticker in sorted_result.index],
})

# Add quartile labels
def get_quartile_label(value, quartiles):
    if value >= quartiles[0.75]:
        return 'Q4'
    elif value >= quartiles[0.5]:
        return 'Q3'
    elif value >= quartiles[0.25]:
        return 'Q2'
    else:
        return 'Q1'

results_df['Quartile'] = results_df['Momentum'].apply(lambda x: get_quartile_label(x, quartiles))

# Print results
print("\n" + "="*80)
print("FILTERED RESULTS - SORTED BY MOMENTUM")
print("="*80)
print(f"{'Ticker':<15} {'Momentum':>12} {'Price':>12} {'Quartile':>10}")
print("-"*80)

for _, row in results_df.iterrows():
    print(f"{row['Ticker']:<15} {row['Momentum']:>11.2%}  {row['Price']:>11.2f}  {row['Quartile']:>10}")

print("="*80)
print(f"\nSUMMARY")
print(f"  Total stocks found: {len(results_df)}")
print(f"  Analysis period: {start_date} to {end_date}")
print(f"\nQUARTILE BREAKPOINTS")
print(f"  Q1 (25th percentile): {quartiles[0.25]:>6.2%}")
print(f"  Q2 (50th percentile): {quartiles[0.5]:>6.2%}")
print(f"  Q3 (75th percentile): {quartiles[0.75]:>6.2%}")
print("="*80)


STOCK MOMENTUM ANALYZER - KOMPAS100
✓ Successfully imported 100 tickers from Kompas100

[1/4] Downloading historical data...
      ✓ Data retrieved for 100 tickers

[2/4] Calculating momentum (20-day lookback)...
      ✓ Momentum calculated for 100 tickers

[3/4] Applying filters...
      - Momentum: 0% to 15%
      - Price: Rp 100 to Rp 1,000
      - Volume: > 50,000
      ✓ 6 stocks passed all filters

[4/4] Sorting and analyzing results...

FILTERED RESULTS - SORTED BY MOMENTUM
Ticker              Momentum        Price   Quartile
--------------------------------------------------------------------------------
HMSP.JK               8.72%       810.00          Q4
SGER.JK               8.61%       454.00          Q4
SMIL.JK               7.83%       358.00          Q3
KPIG.JK               6.51%       180.00          Q2
PWON.JK               5.26%       360.00          Q1
MAPA.JK               2.94%       700.00          Q1

SUMMARY
  Total stocks found: 6
  Analysis period: 2024-01-0